# PathSnap — HMM and Viterbi route reconstruction

The classifier scores each candidate arc **independently**, so consecutive GPS points can be assigned to arcs that are not connected (a jump to a side street and back). This notebook adds a Hidden Markov Model that uses **road-network connectivity** and the **order of observations** to pick one coherent route, decoded with the Viterbi algorithm.

**Classifier used:** HistGradientBoosting (Model 2) — best student_test top-1 of the models compared. Its outputs are used consistently: out-of-fold probabilities on train for tuning the decoder, the saved final model on student_test.

**Before running on Colab**, put both zips in `MyDrive/PathSnap/`:
* `pathsnap_features.zip` — features and labels (same as the model notebooks)
* `pathsnap_clean.zip` — cleaned tracks, road networks and reference routes (new; transitions need the graph)

Runtime on Colab: roughly 5–8 minutes.

## The model

| Part | Choice |
| --- | --- |
| **Hidden state** at observation *t* | one of its candidate arcs, together with the point where the GPS fix projects onto that arc (`proj_t`) |
| **Emission** | the classifier's probability for that candidate: $\log P(o_t \mid s_t) = w \cdot \log p_{clf}(s_t)$ |
| **Transition** | how well the driving distance along the network between the two candidate positions matches the straight-line distance between the two GPS fixes (Newson & Krumm, 2009), with a penalty for reversing onto the same road: $\log P(s_t \mid s_{t-1}) = -\lvert d_{route} - d_{gc} \rvert / \beta \; - \; \lambda_{U}\,[\text{U-turn}]$ |
| **Invalid transition** | no forward path within 2 km of network distance → probability 0 |
| **Decoding** | Viterbi in log space |

Viterbi recurrence, with $\delta_t(j)$ the best log-score of any state sequence ending in candidate $j$ at observation $t$:

$$\delta_t(j) = \log P(o_t \mid j) + \max_i \big[\delta_{t-1}(i) + \log P(j \mid i)\big], \qquad \psi_t(j) = \arg\max_i \big[\delta_{t-1}(i) + \log P(j \mid i)\big]$$

The best final state is $\arg\max_j \delta_T(j)$; the path is recovered by following $\psi$ backwards.

**Special cases (all required by the brief)**
* **GPS gaps** — the chain restarts at every time-gap segment boundary from the cleaning step (gaps > 60 s): no transition is assumed across a gap.
* **Invalid transitions** — if no candidate at *t* is reachable from any candidate at *t−1*, the chain is ended there (a "break") and restarted.
* **Missing candidates** — observations with no candidate within 50 m are skipped by the decoder and then given the arc of the nearest decoded observation in the same segment.
* **Only two parameters matter:** multiplying all log terms by one constant does not change Viterbi's answer, so $w$ is fixed at 1 and only $\beta$ and $\lambda_U$ are tuned — on **train** out-of-fold predictions only.

In [ ]:
# ---- Paths: Colab (Drive zips) or local run --------------------------------------------
import os, sys, zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/PathSnap")
    DATA = Path("/content/pathsnap")
    RESULTS = DRIVE / "results"
    for name, check in [("pathsnap_features.zip", "data_features"), ("pathsnap_clean.zip", "data_clean")]:
        if not (DATA / check).exists():
            with zipfile.ZipFile(DRIVE / name) as z:
                z.extractall(DATA)
else:
    DATA = Path(os.environ.get("PATHSNAP_DATA", "../.."))
    RESULTS = Path(os.environ.get("PATHSNAP_RESULTS", "results"))

FEAT_DIR, CLEAN_DIR = DATA / "data_features", DATA / "data_clean"
OUT = RESULTS / "hmm_viterbi"
OUT.mkdir(parents=True, exist_ok=True)
print("features:", FEAT_DIR, "| clean:", CLEAN_DIR, "| output:", OUT)

In [ ]:
import json, time, itertools, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, joblib
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold

SEED = 42
np.random.seed(SEED)
FEATURES = json.loads((FEAT_DIR / "feature_params.json").read_text())["FEATURES"]

# Classifier: the same configuration as the Model 2 notebook, so out-of-fold train probabilities
# behave like the saved model's student_test probabilities.
CLF_CONFIG = dict(learning_rate=0.1, max_iter=300, max_leaf_nodes=31, min_samples_leaf=20,
                  l2_regularization=0.0, max_bins=255, early_stopping=False, random_state=SEED)

GRID = {"beta": [0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0],     # metres
        "uturn_pen": [0, 5, 10, 20, 40, 80]}                # log-probability penalty
W_EMIT = 1.0
print(f"sklearn {sklearn.__version__}")

## 1. HMM implementation

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra

EARTH_R = 6371008.8
MAX_ROUTE_M = 2000.0   # longest network path searched between consecutive observations


def haversine_m(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, (lon1, lat1, lon2, lat2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * EARTH_R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


class TripGraph:
    """Directed road graph of one trip (canonical arcs only) with cached node-to-node shortest paths."""

    def __init__(self, net):
        self.u, self.v, self.L = net["arc_u"], net["arc_v"], net["arc_length_m"]
        keep = net["arc_is_canonical"]
        n = len(net["node_lon"])
        self.G = csr_matrix((np.maximum(self.L[keep], 0.01), (self.u[keep], self.v[keep])), shape=(n, n))
        self.pair_to_arc = {(a, b): i for i, a, b in zip(np.flatnonzero(keep), self.u[keep], self.v[keep])}
        self.cache = {}
        self.searched = set()

    def node_dist(self, pairs):
        """Shortest distance for (src, dst) node pairs; inf if farther than MAX_ROUTE_M."""
        need = sorted({s for s, _ in pairs} - self.searched)
        for k in range(0, len(need), 64):
            src = need[k:k + 64]
            D = dijkstra(self.G, indices=src, limit=MAX_ROUTE_M)
            for row, s in enumerate(src):
                reach = np.flatnonzero(np.isfinite(D[row]))
                self.cache.update({(s, int(d)): float(D[row, d]) for d in reach})
            self.searched.update(src)
        return np.array([0.0 if s == d else self.cache.get((s, d), np.inf) for s, d in pairs])

    def path_arcs(self, a, b):
        """Arcs strictly between arc a and arc b on the shortest path (for route reconstruction)."""
        s, d = self.v[a], self.u[b]
        if s == d:
            return []
        _, pred = dijkstra(self.G, indices=s, limit=MAX_ROUTE_M, return_predecessors=True)
        if pred[d] < 0:
            return None
        nodes, x = [d], d
        while x != s:
            x = pred[x]
            nodes.append(x)
        nodes = nodes[::-1]
        return [self.pair_to_arc[(p, q)] for p, q in zip(nodes[:-1], nodes[1:])]


def build_steps(cands, obs, graph):
    """Pre-compute everything about transitions that does not depend on tunable parameters.

    cands: rows of one trip with idx, arc_id, proj_t, p (classifier probability)
    obs:   observations of the trip in time order with idx, lon, lat, t, segment_id
    Returns per decodable observation: candidate arcs + log emissions, and per consecutive pair:
    the route-minus-straight-line distance matrix, a U-turn mask and the gap flag.
    """
    g = cands.sort_values(["idx", "p"], ascending=[True, False]).groupby("idx", sort=True)
    by_idx = {i: d for i, d in g}
    seq = [i for i in obs.idx if i in by_idx]          # observations with no candidate are skipped
    pos = obs.set_index("idx").loc[seq]
    u, v, L = graph.u, graph.v, graph.L

    states = [(by_idx[i].arc_id.to_numpy(), by_idx[i].proj_t.to_numpy(), by_idx[i].p.to_numpy()) for i in seq]
    new_segment = np.r_[True, pos.segment_id.to_numpy()[1:] != pos.segment_id.to_numpy()[:-1]]
    gc = np.r_[np.nan, haversine_m(pos.lon.to_numpy()[:-1], pos.lat.to_numpy()[:-1],
                                   pos.lon.to_numpy()[1:], pos.lat.to_numpy()[1:])]

    # all node pairs needed, solved in one batch per trip
    pairs = set()
    for k in range(1, len(seq)):
        if new_segment[k]:
            continue
        a, b = states[k - 1][0], states[k][0]
        pairs.update(zip(np.repeat(v[a], len(b)).tolist(), np.tile(u[b], len(a)).tolist()))
    pairs = list(pairs)
    dist = dict(zip(pairs, graph.node_dist(pairs))) if pairs else {}

    trans = [None]
    for k in range(1, len(seq)):
        if new_segment[k]:
            trans.append(None)
            continue
        (a, ta, _), (b, tb, _) = states[k - 1], states[k]
        A, B = a[:, None], b[None, :]
        nd = np.array([[dist[(v[x], u[y])] for y in b] for x in a])
        route = (1 - ta[:, None]) * L[A] + nd + tb[None, :] * L[B]
        same = A == B
        route = np.where(same, np.abs(tb[None, :] - ta[:, None]) * L[A], route)
        uturn = (u[A] == v[B]) & (v[A] == u[B])
        trans.append((np.abs(route - gc[k]), uturn))
    return {"idx": np.array(seq), "states": states, "trans": trans, "new_segment": new_segment}


def viterbi(steps, beta, w_emit, uturn_pen, eps=1e-6):
    """Returns the chosen arc per decodable observation and the number of breaks."""
    states, trans = steps["states"], steps["trans"]
    n = len(states)
    choice = np.empty(n, dtype=np.int64)
    breaks = 0
    k0 = 0
    score = w_emit * np.log(states[0][2] + eps)
    back = [None]
    for k in range(1, n + 1):
        boundary = k == n or trans[k] is None
        if not boundary:
            diff, uturn = trans[k]
            logT = -diff / beta - uturn_pen * uturn
            tot = score[:, None] + logT
            best = tot.argmax(0)
            new = tot[best, np.arange(tot.shape[1])]
            if not np.isfinite(new).any():        # no feasible transition: break the chain here
                boundary = True
                breaks += 1
            else:
                score = new + w_emit * np.log(states[k][2] + eps)
                back.append(best)
                continue
        # backtrack the finished chain k0 .. k-1
        j = int(np.argmax(score))
        for t in range(k - 1, k0 - 1, -1):
            choice[t] = states[t][0][j]
            if t > k0:
                j = back[t - k0][j]
        if k < n:
            k0, back = k, [None]
            score = w_emit * np.log(states[k][2] + eps)
    return choice, breaks


def classifier_only(steps):
    return np.array([s[0][0] for s in steps["states"]])   # states are sorted by p, highest first

## 2. Load features, cleaned tracks and reference labels
All candidate rows are kept here, including those of unlabelled observations (`label = -1`): the decoder needs every observation's candidates. Labels are only used for tuning and scoring.

In [ ]:
flags = pd.read_csv(CLEAN_DIR / "trip_flags.csv", dtype={"trip_id": str})
TRAIN = sorted(flags[(flags.partition == "train") & flags.use_for_tuning.astype(bool)].trip_id)
TEST = sorted(flags[flags.partition == "student_test"].trip_id)

def load_rows(part, trips):
    out = []
    for t in trips:
        f = pd.read_csv(FEAT_DIR / part / f"{t}.csv.gz", dtype={"trip_id": str})
        f["label"] = pd.read_csv(FEAT_DIR / "labels" / part / f"{t}.csv.gz").label.values
        out.append(f)
    return pd.concat(out, ignore_index=True)

def load_refs(part, trips):
    ref = {}
    for t in trips:
        p = pd.read_csv(FEAT_DIR / "labels" / part / f"{t}_points.csv.gz")
        ref[t] = p[p.labelled].set_index("idx").ref_arc
    return ref

train_rows, test_rows = load_rows("train", TRAIN), load_rows("student_test", TEST)
train_ref, test_ref = load_refs("train", TRAIN), load_refs("student_test", TEST)
print(f"train: {len(TRAIN)} trips, {len(train_rows):,} candidate rows | student_test: {len(TEST)} trips, {len(test_rows):,} rows")

## 3. Classifier probabilities
* **Train:** out-of-fold — each trip is scored by a model trained on the other folds (GroupKFold by trip), exactly as an unseen trip would be. The decoder is tuned on these.
* **Student_test:** the saved final Model 2 (`results/hist_gradient_boosting/hgb_model.joblib`); if it is not there, the same configuration is fitted on all usable train trips.

In [ ]:
t0 = time.time()
train_rows["p"] = np.nan
for tr, va in GroupKFold(n_splits=5).split(train_rows, groups=train_rows.trip_id):
    a = train_rows.iloc[tr]
    a = a[a.label >= 0]
    m = HistGradientBoostingClassifier(**CLF_CONFIG).fit(a[FEATURES], a.label)
    train_rows.loc[train_rows.index[va], "p"] = m.predict_proba(train_rows.iloc[va][FEATURES])[:, 1]
print(f"train out-of-fold probabilities in {time.time() - t0:.0f}s")

saved = RESULTS / "hist_gradient_boosting" / "hgb_model.joblib"
if saved.exists():
    clf = joblib.load(saved)
    print("student_test: using saved Model 2 from", saved)
else:
    a = train_rows[train_rows.label >= 0]
    clf = HistGradientBoostingClassifier(**CLF_CONFIG).fit(a[FEATURES], a.label)
    print("student_test: saved Model 2 not found; fitted the same configuration on all usable train trips")
test_rows["p"] = clf.predict_proba(test_rows[FEATURES])[:, 1]

## 4. Transition tables
For every pair of consecutive observations, the network distance between every candidate pair is computed once (Dijkstra, bounded at 2 km, cached per trip). These tables do not depend on $\beta$ or $\lambda_U$, so tuning only re-runs Viterbi.

In [ ]:
def prepare(part, trips, rows):
    steps, graphs, obs_all = {}, {}, {}
    for t in trips:
        graphs[t] = TripGraph(np.load(CLEAN_DIR / part / t / "network_clean.npz"))
        trk = pd.read_csv(CLEAN_DIR / part / t / "track_clean.csv")
        obs_all[t] = trk[~trk.is_spike & (trk.segment_id >= 0)].reset_index(drop=True)
        steps[t] = build_steps(rows[rows.trip_id == t], obs_all[t], graphs[t])
    return steps, graphs, obs_all

t0 = time.time(); train_steps, train_graphs, train_obs = prepare("train", TRAIN, train_rows)
print(f"train tables: {time.time() - t0:.0f}s")
t0 = time.time(); test_steps, test_graphs, test_obs = prepare("student_test", TEST, test_rows)
print(f"student_test tables: {time.time() - t0:.0f}s")

## 5. Assign every observation, and the evaluation metrics
* **Pointwise top-1** — share of labelled observations whose predicted arc is the reference arc (same definition as the model notebooks; skipped observations are filled first, see below).
* **Route continuity** — share of consecutive observation pairs (within a segment) whose predicted arcs are the same arc or joined by a forward network path no more than 100 m longer than the straight-line distance. A classifier jumping to a side street and back breaks continuity twice.
* **Sequence level** — the predicted route is rebuilt as the ordered arcs the vehicle passes, filling the shortest path between consecutive distinct arcs. It is compared with the original reference arcs (not the gap fills from cleaning) by **length-weighted precision, recall and F1**: how many metres of the predicted route are on the reference route, and how many metres of the reference route were found.

In [ ]:
def assign_all(steps, obs, chosen):
    # every observation gets an arc: skipped ones take the nearest decoded observation in their segment
    s = pd.Series(chosen, index=steps["idx"]).reindex(obs.idx)
    seg = obs.set_index("idx").segment_id
    return s.groupby(seg).transform(lambda x: x.ffill().bfill())

def top1(pred, ref):
    p = pred.reindex(ref.index)
    return int((p == ref).sum()), len(ref)

def continuity(pred, obs, graph):
    o = obs.set_index("idx")
    a, b = pred.to_numpy()[:-1], pred.to_numpy()[1:]
    same_seg = o.segment_id.to_numpy()[:-1] == o.segment_id.to_numpy()[1:]
    ok_rows = same_seg & ~np.isnan(a) & ~np.isnan(b)
    a, b = a[ok_rows].astype(int), b[ok_rows].astype(int)
    gc = haversine_m(o.lon.to_numpy()[:-1][ok_rows], o.lat.to_numpy()[:-1][ok_rows],
                     o.lon.to_numpy()[1:][ok_rows], o.lat.to_numpy()[1:][ok_rows])
    nd = graph.node_dist(list(zip(graph.v[a].tolist(), graph.u[b].tolist())))
    connected = (a == b) | (nd <= gc + 100)
    return int(connected.sum()), len(connected)

def rebuild_route(pred, obs, graph):
    o = obs.set_index("idx")
    arcs, prev, prev_seg = [], None, None
    for i, a in pred.items():
        if np.isnan(a):
            continue
        a = int(a)
        if prev is not None and o.segment_id[i] == prev_seg and a != prev:
            mid = graph.path_arcs(prev, a)
            arcs.extend(mid or [])
        if a != prev:
            arcs.append(a)
        prev, prev_seg = a, o.segment_id[i]
    return arcs

def route_prf(arcs, part, trip, graph):
    ref = pd.read_csv(CLEAN_DIR / "labels" / part / trip / "route_clean.csv")
    ref = set(ref.canonical_arc_id[~ref.is_filled])
    pred = set(arcs)
    L = graph.L
    hit = sum(L[a] for a in pred & ref)
    prec, rec = hit / max(sum(L[a] for a in pred), 1e-9), hit / max(sum(L[a] for a in ref), 1e-9)
    return prec, rec, (2 * prec * rec / (prec + rec) if prec + rec else 0.0)

## 6. Decoder parameter search — train only (out-of-fold probabilities)

In [ ]:
def train_top1(beta, up):
    ok = n = 0
    for t in TRAIN:
        ch, _ = viterbi(train_steps[t], beta, W_EMIT, up)
        k, m = top1(assign_all(train_steps[t], train_obs[t], ch), train_ref[t])
        ok += k; n += m
    return ok / n

t0 = time.time()
search = pd.DataFrame([{"beta": b, "uturn_pen": u, "top1": train_top1(b, u)}
                       for b, u in itertools.product(GRID["beta"], GRID["uturn_pen"])])
best = search.loc[search.top1.idxmax()]
BETA, UTURN = float(best.beta), float(best.uturn_pen)

ok = n = 0
for t in TRAIN:
    k, m = top1(assign_all(train_steps[t], train_obs[t], classifier_only(train_steps[t])), train_ref[t]); ok += k; n += m
print(f"{len(search)} settings in {time.time() - t0:.0f}s | train classifier-only top-1 {ok / n:.4f} "
      f"| best HMM top-1 {best.top1:.4f} at beta={BETA}, uturn_pen={UTURN}")

pv = search.pivot(index="beta", columns="uturn_pen", values="top1")
fig, ax = plt.subplots(figsize=(7, 4.5))
im = ax.imshow(pv.values, cmap="Blues", aspect="auto")
ax.set_xticks(range(len(pv.columns)), pv.columns); ax.set_yticks(range(len(pv.index)), pv.index)
ax.set_xlabel("U-turn penalty"); ax.set_ylabel("beta (m)")
for i in range(pv.shape[0]):
    for j in range(pv.shape[1]):
        ax.text(j, i, f"{pv.values[i, j]:.3f}", ha="center", va="center", fontsize=8,
                color="white" if pv.values[i, j] > pv.values.mean() else "black")
ax.set_title("Train out-of-fold top-1 by decoder setting")
fig.tight_layout(); fig.savefig(OUT / "hmm_parameter_search.png", dpi=150); plt.show()

## 7. Student_test — classifier only vs classifier + HMM
Decoder parameters are fixed from section 6; student_test is only measured.

In [ ]:
rows = []
store = {}
for t in TEST:
    st, ob, g = test_steps[t], test_obs[t], test_graphs[t]
    t0 = time.time(); ch, breaks = viterbi(st, BETA, W_EMIT, UTURN); dec_s = time.time() - t0
    for method, chosen in [("classifier", classifier_only(st)), ("hmm", ch)]:
        pred = assign_all(st, ob, chosen)
        k, m = top1(pred, test_ref[t]); c, cn = continuity(pred, ob, g)
        arcs = rebuild_route(pred, ob, g); pr, rc, f1 = route_prf(arcs, "student_test", t, g)
        rows.append({"trip_id": t, "method": method, "correct": k, "n_obs": m, "connected": c, "pairs": cn,
                     "route_precision": pr, "route_recall": rc, "route_f1": f1,
                     "breaks": breaks if method == "hmm" else np.nan, "decode_s": dec_s if method == "hmm" else np.nan,
                     "skipped_obs": len(ob) - len(st["idx"])})
        store[(t, method)] = pred
res = pd.DataFrame(rows)

def summary(r):
    return pd.Series({"top1": r.correct.sum() / r.n_obs.sum(), "continuity": r.connected.sum() / r.pairs.sum(),
                      "route_precision": r.route_precision.mean(), "route_recall": r.route_recall.mean(),
                      "route_f1": r.route_f1.mean(), "breaks": r.breaks.sum(), "decode_s_total": r.decode_s.sum()})
overall = res.groupby("method").apply(summary).T
overall.round(4)

In [ ]:
per_trip = res.assign(top1=res.correct / res.n_obs, continuity=res.connected / res.pairs).pivot(
    index="trip_id", columns="method", values=["top1", "continuity", "route_f1"])
per_trip.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
pt = res.assign(top1=res.correct / res.n_obs).pivot(index="trip_id", columns="method", values="top1").sort_values("classifier")
x = np.arange(len(pt))
axes[0].bar(x - 0.2, pt.classifier, 0.4, label="classifier only", color="#eb6834")
axes[0].bar(x + 0.2, pt.hmm, 0.4, label="classifier + HMM", color="#2a78d6")
axes[0].set_xticks(x, [t[-2:] for t in pt.index]); axes[0].set_xlabel("student_test trip (last two digits)")
axes[0].set_ylim(0.6, 1.0); axes[0].set_ylabel("pointwise top-1"); axes[0].set_title("Top-1 per trip")
m = ["top1", "continuity", "route_precision", "route_recall", "route_f1"]
axes[1].bar(np.arange(len(m)) - 0.2, overall.loc[m, "classifier"], 0.4, color="#eb6834", label="classifier only")
axes[1].bar(np.arange(len(m)) + 0.2, overall.loc[m, "hmm"], 0.4, color="#2a78d6", label="classifier + HMM")
axes[1].set_xticks(range(len(m)), ["top-1", "continuity", "route\nprecision", "route\nrecall", "route F1"])
axes[1].set_ylim(0.6, 1.0); axes[1].set_title("Student_test summary")
fig.legend(*axes[0].get_legend_handles_labels(), loc="lower center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0.07, 1, 1)); fig.savefig(OUT / "hmm_vs_classifier.png", dpi=150); plt.show()

## 8. Save the decoder configuration, results and predictions
* `hmm_config.json` — everything the API needs to reproduce decoding: parameters, limits, classifier used.
* `hmm_results.json` — parameter search, student_test summary and per-trip results.
* `student_test_route_predictions.csv.gz` — one row per observation: classifier arc and HMM arc.

In [ ]:
config = {"classifier": "HistGradientBoosting (Model 2)", "classifier_config": CLF_CONFIG,
          "beta_m": BETA, "uturn_penalty": UTURN, "emission_weight": W_EMIT, "max_route_m": MAX_ROUTE_M,
          "gap_handling": "restart at time-gap segment boundaries (gaps > 60 s, from cleaning)",
          "invalid_transition": "chain ended and restarted (break)",
          "missing_candidates": "skipped by decoder, then given the arc of the nearest decoded observation in the segment",
          "tuned_on": "train out-of-fold probabilities only", "versions": {"python": platform.python_version(), "sklearn": sklearn.__version__}}
(OUT / "hmm_config.json").write_text(json.dumps(config, indent=2))
(OUT / "hmm_results.json").write_text(json.dumps({
    "parameter_search": search.to_dict(orient="records"),
    "student_test_summary": overall.to_dict(),
    "student_test_per_trip": res.to_dict(orient="records")}, indent=2, default=float))
pd.concat([pd.DataFrame({"trip_id": t, "idx": store[(t, "classifier")].index,
                         "arc_classifier": store[(t, "classifier")].values, "arc_hmm": store[(t, "hmm")].values})
           for t in TEST]).to_csv(OUT / "student_test_route_predictions.csv.gz", index=False)
print("saved to", OUT)

## 9. Map — classifier only vs HMM on one trip
Green = reference route, orange = classifier-only arcs, blue = HMM route; GPS points are blue where the HMM is right and red where it is wrong. Toggle layers top right.

In [ ]:
import folium

TRIP = "00000077"   # any student_test trip: 00000000 07 09 40 52 56 65 76 77 91

net = np.load(CLEAN_DIR / "student_test" / TRIP / "network_clean.npz")
nlon, nlat, au, av = net["node_lon"], net["node_lat"], net["arc_u"], net["arc_v"]
ob = test_obs[TRIP].set_index("idx")
line = lambda a: [[nlat[au[a]], nlon[au[a]]], [nlat[av[a]], nlon[av[a]]]]

m = folium.Map(location=[ob.lat.mean(), ob.lon.mean()], zoom_start=14, tiles="OpenStreetMap")
ref = pd.read_csv(CLEAN_DIR / "labels" / "student_test" / TRIP / "route_clean.csv")
layers = [("Reference route", ref.canonical_arc_id[~ref.is_filled], "green", 8, 0.4),
          ("Classifier only", store[(TRIP, "classifier")].dropna().astype(int).unique(), "#eb6834", 3, 0.9),
          ("HMM route", rebuild_route(store[(TRIP, "hmm")], test_obs[TRIP], test_graphs[TRIP]), "#2a78d6", 4, 0.9)]
for name, arcs, col, w, op in layers:
    fg = folium.FeatureGroup(name).add_to(m)
    for a in pd.unique(np.asarray(arcs)):
        folium.PolyLine(line(a), color=col, weight=w, opacity=op).add_to(fg)
pts = folium.FeatureGroup("GPS points (blue = HMM right, red = wrong)").add_to(m)
hm, rf = store[(TRIP, "hmm")], test_ref[TRIP]
for i, r in ob.iterrows():
    ok = i in rf.index and hm.get(i) == rf[i]
    folium.CircleMarker([r.lat, r.lon], radius=2, weight=0, fill=True, fill_opacity=0.8,
                        color="#2a78d6" if ok else "#d62728", tooltip=f"idx {i}").add_to(pts)
folium.LayerControl(collapsed=False).add_to(m)
m.fit_bounds([[ob.lat.min(), ob.lon.min()], [ob.lat.max(), ob.lon.max()]])
r = res[res.trip_id == TRIP].set_index("method")
print(f"Trip {TRIP}: top-1 classifier {r.correct.classifier / r.n_obs.classifier:.1%} -> HMM {r.correct.hmm / r.n_obs.hmm:.1%}")
m.save(str(OUT / f"map_hmm_{TRIP}.html"))
m